<a href="https://colab.research.google.com/github/karthik-srivathsa-05/flyrank-ai/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karthik-srivathsa-05/flyrank-ai/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

# ML-08 — Capstone Modeling Lane

I use Logistic Regression because this lane is a binary decline-detection problem and the output needs to be interpretable for a content-review queue. The model estimates the probability that a page will experience a meaningful performance decline in March based only on information available by the end of February. I use the same February feature vector established in ML-05 and keep client and content identifiers only for grouping and reporting, not as model features.

The target is a binary proxy: a page is labelled 1 when its March performance falls by at least 20% relative to its February baseline; otherwise it is labelled 0.

The model is intended as decision support rather than a causal model. A high predicted probability means that the observed February signals are associated with a higher likelihood of the defined March decline in this dataset; it does not prove that a content refresh would cause performance to improve.

In [8]:
REL = "hf://datasets/FlyRank/internship-warehouse"

FACT_FEB = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-02/*.parquet'"
    f")"
)

FACT_MAR = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'"
    f")"
)

DIM_CONTENT = (
    f"read_parquet('{REL}/dim_content.parquet')"
)

print("February and March sources defined.")

February and March sources defined.


In [9]:
import os
import getpass
import duckdb
import pandas as pd
import numpy as np

def get_hf_token():
    token = os.environ.get("HF_TOKEN")

    if token:
        return token

    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")

        if token:
            return token
    except Exception:
        pass

    return getpass.getpass("Enter Hugging Face READ token: ")


HF_TOKEN = get_hf_token()

if not HF_TOKEN:
    raise ValueError("HF_TOKEN was not provided.")

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

try:
    con.execute("INSTALL httpfs")
except Exception:
    pass

con.execute("LOAD httpfs")

con.execute(
    "CREATE OR REPLACE SECRET hf "
    f"(TYPE HUGGINGFACE, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

FACT_FEB = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-02/*.parquet'"
    f")"
)

FACT_MAR = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'"
    f")"
)

DIM_CONTENT = (
    f"read_parquet('{REL}/dim_content.parquet')"
)

DIM_CLIENTS = (
    f"read_parquet('{REL}/dim_clients.parquet')"
)

print("DuckDB version:", con.sql("SELECT version()").fetchone()[0])
print("httpfs loaded successfully.")
print("Hugging Face authentication configured.")
print("February feature data path configured.")
print("March label data path configured.")

DuckDB version: v1.3.2
httpfs loaded successfully.
Hugging Face authentication configured.
February feature data path configured.
March label data path configured.


In [10]:
feb_check = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_report_date,
    MAX(report_date) AS max_report_date
FROM {FACT_FEB}
""").df()

display(feb_check)

,row_count,min_report_date,max_report_date
0,7355108,2026-02-01,2026-02-28


In [12]:
model_frame = con.sql(f"""
WITH feb AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(COALESCE(gsc_impressions, 0)) AS impressions_30d,
        SUM(COALESCE(gsc_clicks, 0)) AS clicks_30d,

        AVG(
            CASE
                WHEN gsc_avg_position IS NOT NULL
                THEN gsc_avg_position
            END
        ) AS avg_position,

        COUNT(
            CASE
                WHEN COALESCE(gsc_impressions, 0) > 0
                THEN 1
            END
        ) AS days_with_impressions

    FROM {FACT_FEB}

    WHERE report_date BETWEEN DATE '2026-02-01'
                          AND DATE '2026-02-28'

    GROUP BY
        client_hash_id,
        content_hash_id
),

march AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(COALESCE(gsc_clicks, 0)) AS march_clicks

    FROM {FACT_MAR}

    WHERE report_date BETWEEN DATE '2026-03-01'
                          AND DATE '2026-03-31'

    GROUP BY
        client_hash_id,
        content_hash_id
),

feb_baseline AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(COALESCE(gsc_clicks, 0)) AS feb_clicks

    FROM {FACT_FEB}

    WHERE report_date BETWEEN DATE '2026-02-01'
                          AND DATE '2026-02-28'

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    f.client_hash_id,
    f.content_hash_id,

    f.impressions_30d,
    f.clicks_30d,
    f.avg_position,
    f.days_with_impressions,

    COALESCE(m.march_clicks, 0) AS march_clicks,
    COALESCE(b.feb_clicks, 0) AS feb_clicks,

    CASE
        WHEN COALESCE(b.feb_clicks, 0) > 0
             AND COALESCE(m.march_clicks, 0)
                 < 0.70 * b.feb_clicks
        THEN 1
        ELSE 0
    END AS decline_label

FROM feb f

LEFT JOIN march m
    ON f.client_hash_id = m.client_hash_id
   AND f.content_hash_id = m.content_hash_id

LEFT JOIN feb_baseline b
    ON f.client_hash_id = b.client_hash_id
   AND f.content_hash_id = b.content_hash_id

WHERE COALESCE(b.feb_clicks, 0) > 0
""").df()

print("Modeling rows:", len(model_frame))
print("Columns:", list(model_frame.columns))

display(model_frame.head())

Modeling rows: 55090
Columns: ['client_hash_id', 'content_hash_id', 'impressions_30d', 'clicks_30d', 'avg_position', 'days_with_impressions', 'march_clicks', 'feb_clicks', 'decline_label']


,client_hash_id,content_hash_id,impressions_30d,clicks_30d,avg_position,days_with_impressions,march_clicks,feb_clicks,decline_label
0,client_3ffa76342f366962,content_1546aabff77c05a4,5.0,1.0,5.000000,4,0.0,1.0,1
1,client_3ffa76342f366962,content_bdff81f402e40680,2.0,1.0,28.000000,2,0.0,1.0,1
2,client_3ffa76342f366962,content_70a45790a6dc6ec4,5.0,1.0,4.416667,2,0.0,1.0,1
3,client_3ffa76342f366962,content_6f10d77a6510c764,1.0,1.0,0.000000,1,0.0,1.0,1
4,client_3ffa76342f366962,content_0674cc4ae0f68a90,74.0,1.0,8.063910,19,0.0,1.0,1


### Method choice

I use **Logistic Regression** because this is a binary decline-classification problem and the goal is an interpretable baseline model rather than maximum complexity. Logistic Regression provides a probability-like score that can be used to rank pages for review, while its coefficients provide a simple way to inspect which observed February signals are associated with the March decline label.

The model uses only information available at the February 28, 2026 decision cutoff. The target is whether March clicks fall by more than 30% relative to the February baseline.


In [13]:
FEATURES = [
    "impressions_30d",
    "clicks_30d",
    "avg_position",
    "days_with_impressions"
]

TARGET = "decline_label"
GROUP = "client_hash_id"

X = model_frame[FEATURES].copy()
y = model_frame[TARGET].copy()
groups = model_frame[GROUP].copy()

print("Features:")
for feature in FEATURES:
    print("-", feature)

print("\nTarget:", TARGET)

print("\nFeature matrix shape:", X.shape)
print("Target shape:", y.shape)

print("\nClass distribution:")
display(
    y.value_counts()
      .rename_axis("decline_label")
      .reset_index(name="row_count")
      .assign(
          percentage=lambda df:
              100 * df["row_count"] / len(y)
      )
)

Features:
- impressions_30d
- clicks_30d
- avg_position
- days_with_impressions

Target: decline_label

Feature matrix shape: (55090, 4)
Target shape: (55090,)

Class distribution:


,decline_label,row_count,percentage
0,0,33376,60.584498
1,1,21714,39.415502


### Split design

I use a **grouped train/test split by client**. All pages belonging to the same client remain in either the training set or the test set, rather than being split across both. This reduces the risk that client-specific patterns are learned from one page and evaluated on another page from the same client.

I use a fixed random seed so the split is reproducible. The test set is kept separate until model evaluation, and no March outcome variable is used as a model feature.


In [16]:
from sklearn.model_selection import GroupShuffleSplit

print("GroupShuffleSplit imported successfully.")
RANDOM_STATE = 42
TEST_SIZE = 0.20

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

groups_train = groups.iloc[train_idx].copy()
groups_test = groups.iloc[test_idx].copy()

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

print("\nTraining clients:", groups_train.nunique())
print("Test clients:", groups_test.nunique())

overlap = set(groups_train.unique()).intersection(
    set(groups_test.unique())
)

print("\nClient overlap:", len(overlap))

assert len(overlap) == 0, "Client leakage detected!"

print("Grouped split check: PASSED")

GroupShuffleSplit imported successfully.
Training rows: 34860
Test rows: 20230

Training clients: 32
Test clients: 9

Client overlap: 0
Grouped split check: PASSED


## 3. Train + compare vs my baseline

I train Logistic Regression using the five February features. Missing numeric values are median-imputed using the training data, followed by standardization. The model produces a probability of the defined March decline. For evaluation, the model and the Week-4 rule-based baseline are evaluated on exactly the same grouped test observations. ROC-AUC is the main discrimination metric because it is less dependent on the chosen probability threshold; average precision is also reported because the positive class may not be balanced.


In [18]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000,
            random_state=RANDOM_STATE
        )
    )
])

model.fit(X_train, y_train)

print("Logistic Regression trained successfully.")

Logistic Regression trained successfully.


In [19]:
model_probability = model.predict_proba(X_test)[:, 1]

model_prediction = (
    model_probability >= 0.50
).astype(int)

print("Predictions generated.")
print("Prediction count:", len(model_prediction))
print("Positive predictions:", model_prediction.sum())

Predictions generated.
Prediction count: 20230
Positive predictions: 1774


In [20]:
baseline_test = model_frame.iloc[test_idx].copy()

baseline_test["click_signal"] = (
    1 - baseline_test["clicks_30d"].rank(
        pct=True,
        method="average"
    )
)

baseline_test["position_signal"] = (
    baseline_test["avg_position"].rank(
        pct=True,
        method="average"
    )
)

baseline_test["coverage_signal"] = (
    1 - baseline_test["days_with_impressions"].rank(
        pct=True,
        method="average"
    )
)

# Transparent weighted baseline score
baseline_test["baseline_score"] = (
    0.50 * baseline_test["click_signal"]
    + 0.30 * baseline_test["position_signal"]
    + 0.20 * baseline_test["coverage_signal"]
)

print("Baseline score created.")

display(
    baseline_test[
        [
            "client_hash_id",
            "content_hash_id",
            "clicks_30d",
            "avg_position",
            "days_with_impressions",
            "baseline_score",
            "decline_label"
        ]
    ]
    .sort_values(
        "baseline_score",
        ascending=False
    )
    .head(10)
)

Baseline score created.


,client_hash_id,content_hash_id,clicks_30d,avg_position,days_with_impressions,baseline_score,decline_label
38638,client_3197e6291363b4db,content_91e840b39a7ba247,1.0,47.000000,3,0.926120,1
11097,client_3197e6291363b4db,content_352d9fcaa95cc585,1.0,45.000000,2,0.925996,1
11105,client_3197e6291363b4db,content_bc20268efe2201b3,1.0,61.541667,5,0.924909,1
36985,client_9958f0a7ae1df715,content_718ab3123d9e76d5,1.0,58.200000,5,0.924790,1
25039,client_20259bd6705d81d4,content_f4aa2b83a36d872b,1.0,40.243651,4,0.924449,1
13812,client_ff644d8251367cbb,content_77e00d6e003b5f37,1.0,37.000000,3,0.924414,1
9553,client_9958f0a7ae1df715,content_6f3656a852e308ce,1.0,34.333333,3,0.923450,1
50086,client_73cda7b4e4f265ea,content_eb9b7ad63cea7fed,1.0,48.416667,6,0.922294,1
51901,client_20259bd6705d81d4,content_be85da0d3d2fe07f,1.0,49.519018,7,0.920875,1
11102,client_3197e6291363b4db,content_1ef322975cdd9cf3,1.0,47.211458,8,0.919990,1


In [21]:
from sklearn.metrics import roc_auc_score, average_precision_score

y_test_array = y_test.to_numpy()

model_auc = roc_auc_score(
    y_test_array,
    model_probability
)

model_ap = average_precision_score(
    y_test_array,
    model_probability
)

baseline_probability = (
    baseline_test["baseline_score"].to_numpy()
)

baseline_auc = roc_auc_score(
    y_test_array,
    baseline_probability
)

baseline_ap = average_precision_score(
    y_test_array,
    baseline_probability
)

results = pd.DataFrame({
    "method": [
        "Week-4 baseline",
        "Logistic Regression"
    ],
    "ROC_AUC": [
        baseline_auc,
        model_auc
    ],
    "Average_Precision": [
        baseline_ap,
        model_ap
    ]
})

display(
    results.style.format({
        "ROC_AUC": "{:.4f}",
        "Average_Precision": "{:.4f}"
    })
)

print(f"Baseline ROC-AUC: {baseline_auc:.4f}")
print(f"Model ROC-AUC:    {model_auc:.4f}")
print(f"AUC difference:   {model_auc - baseline_auc:+.4f}")

print(f"\nBaseline Average Precision: {baseline_ap:.4f}")
print(f"Model Average Precision:    {model_ap:.4f}")
print(
    f"AP difference:              "
    f"{model_ap - baseline_ap:+.4f}"
)

,method,ROC_AUC,Average_Precision
0,Week-4 baseline,0.6194,0.4704
1,Logistic Regression,0.6286,0.4721


Baseline ROC-AUC: 0.6194
Model ROC-AUC:    0.6286
AUC difference:   +0.0092

Baseline Average Precision: 0.4704
Model Average Precision:    0.4721
AP difference:              +0.0017


In [22]:
base_rate = y_test.mean()

majority_class_accuracy = max(
    y_test.mean(),
    1 - y_test.mean()
)

print(
    f"Decline base rate: "
    f"{base_rate * 100:.2f}%"
)

print(
    f"Majority-class accuracy baseline: "
    f"{majority_class_accuracy * 100:.2f}%"
)

Decline base rate: 36.25%
Majority-class accuracy baseline: 63.75%


## 4. Errors and interpretation

The model correctly classified 12,950 of 20,230 test observations, with 6,420 false negatives and 860 false positives at the 0.50 classification threshold. The large number of false negatives shows that a simple binary cutoff is not sufficient for identifying every observed decline, so the model is more useful as a ranking signal than as a hard yes/no decision rule.

The largest model coefficient by absolute value was `impressions_30d` (-0.575), followed by `avg_position` (0.195). This indicates that, within this fitted model and feature set, lower February impression volume and weaker average position were associated with higher estimated decline probability. These are measured associations in this dataset, not evidence that either signal causes future decline.

The Logistic Regression model achieved ROC-AUC 0.6286 compared with 0.6194 for the Week-4 baseline, an improvement of 0.0092. Average Precision improved only from 0.4704 to 0.4721. The small improvement suggests that the model adds some directional ranking value over the transparent baseline, but it does not justify claiming a large predictive advantage.


In [23]:
error_frame = model_frame.iloc[test_idx].copy()

error_frame["model_probability"] = model_probability
error_frame["model_prediction"] = model_prediction

error_frame["error_type"] = np.select(
    [
        (error_frame["model_prediction"] == 1)
        & (error_frame["decline_label"] == 0),

        (error_frame["model_prediction"] == 0)
        & (error_frame["decline_label"] == 1)
    ],
    [
        "false_positive",
        "false_negative"
    ],
    default="correct"
)

print("Error counts:")

display(
    error_frame["error_type"]
    .value_counts()
    .rename_axis("error_type")
    .reset_index(name="row_count")
)

Error counts:


,error_type,row_count
0,correct,12950
1,false_negative,6420
2,false_positive,860


In [24]:
false_positives = (
    error_frame[
        error_frame["error_type"] == "false_positive"
    ]
    .sort_values(
        "model_probability",
        ascending=False
    )
)

print("Top false positives:")

display(
    false_positives[
        [
            "client_hash_id",
            "content_hash_id",
            "impressions_30d",
            "clicks_30d",
            "avg_position",
            "days_with_impressions",
            "model_probability",
            "decline_label"
        ]
    ].head(10)
)

Top false positives:


,client_hash_id,content_hash_id,impressions_30d,clicks_30d,avg_position,days_with_impressions,model_probability,decline_label
13776,client_ff644d8251367cbb,content_743afd9c6979d5aa,1041.0,1.0,68.942925,28,0.735027,0
22372,client_73cda7b4e4f265ea,content_150cd9a32fa9b19b,23.0,1.0,54.643590,13,0.724994,0
9270,client_9958f0a7ae1df715,content_7d1731ca239131de,119.0,1.0,54.442857,24,0.698493,0
24856,client_20259bd6705d81d4,content_65cb540e511424c8,290.0,1.0,47.947945,10,0.695945,0
48925,client_b10cb2997d0c7c86,content_aa8574069fd0ea3f,975.0,1.0,58.646900,28,0.690613,0
22581,client_73cda7b4e4f265ea,content_6055eadb7b99f27b,141.0,1.0,51.651184,27,0.678256,0
36900,client_9958f0a7ae1df715,content_dade093b61df1b5c,163.0,1.0,52.184283,28,0.678016,0
38583,client_3197e6291363b4db,content_c5c04445dec4a946,102.0,1.0,46.613194,24,0.661845,0
25001,client_20259bd6705d81d4,content_fff3e915ca28915b,52.0,1.0,38.566667,8,0.661286,0
13830,client_ff644d8251367cbb,content_cc25951cb7b0d5e4,385.0,1.0,43.666713,19,0.652738,0


In [25]:
false_negatives = (
    error_frame[
        error_frame["error_type"] == "false_negative"
    ]
    .sort_values(
        "model_probability",
        ascending=False
    )
)

print("Top false negatives:")

display(
    false_negatives[
        [
            "client_hash_id",
            "content_hash_id",
            "impressions_30d",
            "clicks_30d",
            "avg_position",
            "days_with_impressions",
            "model_probability",
            "decline_label"
        ]
    ].head(10)
)

Top false negatives:


,client_hash_id,content_hash_id,impressions_30d,clicks_30d,avg_position,days_with_impressions,model_probability,decline_label
52469,client_20259bd6705d81d4,content_0557d8e32525be06,213.0,4.0,9.376831,10,0.499923,1
39740,client_73cda7b4e4f265ea,content_5031efe7e0d76d76,14.0,1.0,8.074074,9,0.499769,1
51436,client_fef1a8f436438636,content_80b1e55d460a175e,328.0,3.0,12.807340,16,0.499723,1
52300,client_20259bd6705d81d4,content_bed63bdb1c96dea0,73.0,1.0,8.334182,9,0.499690,1
51971,client_20259bd6705d81d4,content_cf8e7f2c733f0631,207.0,2.0,9.367199,10,0.499539,1
38614,client_3197e6291363b4db,content_5d219151553c0561,20.0,1.0,10.452381,14,0.499467,1
10692,client_73cda7b4e4f265ea,content_ef74180131d04591,67.0,1.0,15.481548,24,0.499459,1
26980,client_0fa64a184f18a4a0,content_60e7e5b0c2fb633f,274.0,1.0,6.801070,4,0.499345,1
55022,client_0fa64a184f18a4a0,content_e773a1d902a80a97,51.0,1.0,6.233853,5,0.499292,1
36849,client_9958f0a7ae1df715,content_9ccb7c702a02d0c1,1133.0,2.0,22.295458,28,0.499289,1


In [26]:
classifier = model.named_steps["classifier"]

coefficients = pd.DataFrame({
    "feature": FEATURES,
    "coefficient": classifier.coef_[0]
})

coefficients["absolute_coefficient"] = (
    coefficients["coefficient"].abs()
)

coefficients = coefficients.sort_values(
    "absolute_coefficient",
    ascending=False
)

print("Logistic Regression coefficients:")

display(
    coefficients[
        [
            "feature",
            "coefficient"
        ]
    ]
)

Logistic Regression coefficients:


,feature,coefficient
0,impressions_30d,-0.575001
2,avg_position,0.195333
3,days_with_impressions,-0.079931
1,clicks_30d,0.030008


In [27]:
summary = pd.DataFrame({
    "Metric": [
        "ROC-AUC",
        "Average Precision"
    ],
    "Week-4 Baseline": [
        baseline_auc,
        baseline_ap
    ],
    "Logistic Regression": [
        model_auc,
        model_ap
    ]
})

summary["Model minus Baseline"] = (
    summary["Logistic Regression"]
    - summary["Week-4 Baseline"]
)

display(
    summary.style.format({
        "Week-4 Baseline": "{:.4f}",
        "Logistic Regression": "{:.4f}",
        "Model minus Baseline": "{:+.4f}"
    })
)

,Metric,Week-4 Baseline,Logistic Regression,Model minus Baseline
0,ROC-AUC,0.6194,0.6286,+0.0092
1,Average Precision,0.4704,0.4721,+0.0017


In [28]:
print("========== W5 MODEL SANITY CHECK ==========")

print(f"Modeling rows: {len(model_frame):,}")
print(f"Training rows: {len(X_train):,}")
print(f"Test rows:     {len(X_test):,}")

print(f"\nDecline base rate: {base_rate:.4f}")

print("\nMODEL VS BASELINE")
print(f"Baseline ROC-AUC: {baseline_auc:.4f}")
print(f"Model ROC-AUC:    {model_auc:.4f}")
print(f"AUC improvement:  {model_auc - baseline_auc:+.4f}")

print(f"\nBaseline AP:      {baseline_ap:.4f}")
print(f"Model AP:         {model_ap:.4f}")
print(f"AP improvement:   {model_ap - baseline_ap:+.4f}")

print("\nLEAKAGE CHECK")
print("Features used:")
for feature in FEATURES:
    print(" -", feature)

for forbidden in [
    "march_clicks",
    "feb_clicks",
    "decline_label"
]:
    assert forbidden not in FEATURES

print("\nNo future/label-derived fields used as model features.")

print("\nGROUP CHECK")
print("Client overlap:", len(overlap))
assert len(overlap) == 0

print("\nW5 SANITY CHECK: PASSED")

========== W5 MODEL SANITY CHECK ==========
Modeling rows: 55,090
Training rows: 34,860
Test rows:     20,230

Decline base rate: 0.3625

MODEL VS BASELINE
Baseline ROC-AUC: 0.6194
Model ROC-AUC:    0.6286
AUC improvement:  +0.0092

Baseline AP:      0.4704
Model AP:         0.4721
AP improvement:   +0.0017

LEAKAGE CHECK
Features used:
 - impressions_30d
 - clicks_30d
 - avg_position
 - days_with_impressions

No future/label-derived fields used as model features.

GROUP CHECK
Client overlap: 0

W5 SANITY CHECK: PASSED


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.